In [ ]:
from pathlib import Path
import gc
import time
import shutil

import numpy as np
import polars as pl
from tqdm import tqdm
from xgboost import XGBRFRegressor
from google.colab import drive


In [ ]:
# Paths
drive.mount("/content/drive")

# Change this if your Google Drive folder name/path is different
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Thesis_Code/limits-to-arbitrage-machine-learning")

DRIVE_PROCESSED = DRIVE_PROJECT_ROOT / "data" / "processed"
DRIVE_OUTPUT = DRIVE_PROJECT_ROOT / "output"
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

DRIVE_PANEL_PATH = DRIVE_PROCESSED / "gkx_master_panel.parquet"

print("Drive panel path:", DRIVE_PANEL_PATH)
print("Exists on Drive?", DRIVE_PANEL_PATH.exists())

if not DRIVE_PANEL_PATH.exists():
    raise FileNotFoundError(f"Panel not found on Drive: {DRIVE_PANEL_PATH}")

# Copy the parquet from Drive to Colab local disk for faster reading
LOCAL_PROCESSED = Path("/content/data/processed")
LOCAL_PROCESSED.mkdir(parents=True, exist_ok=True)

LOCAL_PANEL_PATH = LOCAL_PROCESSED / "gkx_master_panel.parquet"

if (
    not LOCAL_PANEL_PATH.exists()
    or LOCAL_PANEL_PATH.stat().st_size != DRIVE_PANEL_PATH.stat().st_size
):
    print("Copying panel from Drive to Colab local disk...")
    shutil.copy2(DRIVE_PANEL_PATH, LOCAL_PANEL_PATH)

print("Local panel path:", LOCAL_PANEL_PATH)
print("Exists locally?", LOCAL_PANEL_PATH.exists())
print("Local size GB:", LOCAL_PANEL_PATH.stat().st_size / 1e9)

# Use local parquet for input speed, but save outputs to Drive
PANEL_PATH = LOCAL_PANEL_PATH
OUTPUT = DRIVE_OUTPUT


In [ ]:
CHAR_COLS = [
    "absacc", "acc", "aeavol", "age", "agr",
    "baspread", "beta", "betasq", "bm", "bm_ia",
    "cash", "cashdebt", "cashpr", "cfp", "cfp_ia",
    "chatoia", "chcsho", "chempia", "chinv", "chmom",
    "chpmia", "chtx", "cinvest", "convind", "currat",
    "depr", "divi", "divo", "dolvol", "dy",
    "ear", "egr", "ep", "gma", "grcapx",
    "grltnoa", "herf", "hire", "idiovol", "ill",
    "indmom", "invest", "lev", "lgr", "maxret",
    "mom12m", "mom1m", "mom36m", "mom6m", "ms",
    "mvel1", "mve_ia", "nincr", "operprof", "orgcap",
    "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale", "pchquick",
    "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
    "pricedelay", "ps", "quick", "rd", "rd_mve",
    "rd_sale", "realestate", "retvol", "roaq", "roavol",
    "roeq", "roic", "rsup", "salecash", "saleinv",
    "salerec", "secured", "securedind", "sgr", "sin",
    "sp", "std_dolvol", "std_turn", "stdacc", "stdcf",
    "tang", "tb", "turn", "zerotrade"
]

# GKX-style 920 feature set = 94 rank-transformed characteristics × 9 macro states,
# where the 9 macro states are: constant + 8 Goyal-Welch macro variables.
MACRO_NAMES = ["dp", "ep", "bm", "ntis", "tbl", "tms", "dfy", "svar"]

# Because the firm characteristics also include bm and ep, your merged panel may have
# the Goyal macro bm/ep stored with suffixes such as bm_right and ep_right.
# Add extra candidate names here if your schema uses different names.
MACRO_CANDIDATES = {
    "dp":   ["dp", "dp_macro", "dp_goyal", "dp_right", "dp_y", "dp_ts"],
    "ep":   ["ep_macro", "ep_goyal", "ep_right", "ep_y", "ep_ts", "ep_gw"],
    "bm":   ["bm_macro", "bm_goyal", "bm_right", "bm_y", "bm_ts", "bm_gw"],
    "ntis": ["ntis", "ntis_macro", "ntis_goyal", "ntis_right", "ntis_y", "ntis_ts"],
    "tbl":  ["tbl", "tbl_macro", "tbl_goyal", "tbl_right", "tbl_y", "tbl_ts"],
    "tms":  ["tms", "tms_macro", "tms_goyal", "tms_right", "tms_y", "tms_ts"],
    "dfy":  ["dfy", "dfy_macro", "dfy_goyal", "dfy_right", "dfy_y", "dfy_ts"],
    "svar": ["svar", "svar_macro", "svar_goyal", "svar_right", "svar_y", "svar_ts"],
}

MODEL_NAME = "RF_920_fast"
TARGET_MODE = "next_month"

# Smoke test first. After it works, change to range(1987, 2017).
TEST_YEARS = range(1987, 1992)
# TEST_YEARS = range(1987, 2017)

START_TARGET_MONTH_ID = 1957 * 12 + 3
END_TARGET_MONTH_ID = 2016 * 12 + 12


In [ ]:
# Helper functions
def oos_r2(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    denom = np.sum(y_true ** 2)
    if denom == 0:
        return np.nan

    return 1.0 - np.sum((y_true - y_pred) ** 2) / denom


def resolve_macro_cols(schema_names):
    """Return a dict mapping GKX macro names to actual column names in your panel."""
    schema_names = set(schema_names)
    resolved = {}
    missing = []

    for macro_name in MACRO_NAMES:
        found = None
        for candidate in MACRO_CANDIDATES[macro_name]:
            if candidate in schema_names:
                found = candidate
                break
        if found is None:
            missing.append(macro_name)
        else:
            resolved[macro_name] = found

    if missing:
        possible = sorted([
            c for c in schema_names
            if any(m in c.lower() for m in ["dp", "ep", "bm", "ntis", "tbl", "tms", "dfy", "svar"])
        ])
        raise ValueError(
            "Could not resolve these macro columns: " + str(missing) + "\\n"
            "Relevant-looking columns in the panel are:\\n" + str(possible[:100]) + "\\n\\n"
            "Fix: add your actual macro column names to MACRO_CANDIDATES. "
            "For bm and ep, do not use the firm characteristic bm/ep by accident."
        )

    return resolved


def prepare_panel():
    lf = pl.scan_parquet(PANEL_PATH)
    schema = lf.collect_schema()
    schema_names = list(schema.keys())

    missing_chars = [c for c in CHAR_COLS if c not in schema_names]
    if missing_chars:
        raise ValueError(f"Missing characteristic columns: {missing_chars[:20]}")

    macro_cols = resolve_macro_cols(schema_names)
    print("Resolved macro columns:", macro_cols)

    lf = lf.with_columns([
        pl.col("month_id").cast(pl.Int64),
        pl.col("month").dt.year().alias("info_year"),
    ])

    # next-month target: information at month t predicts return in month t+1
    y_lf = lf.select([
        "permno",
        (pl.col("month_id") - 1).alias("month_id"),
        pl.col("ret_excess").alias("y"),
        pl.col("month").alias("target_month"),
        pl.col("month_id").alias("target_month_id"),
        pl.col("month").dt.year().alias("target_year"),
    ])

    lf = lf.join(y_lf, on=["permno", "month_id"], how="left")

    lf = lf.with_columns(
        pl.col("mktcap").alias("size_for_ranking")
    )

    lf = lf.filter(
        (pl.col("target_month_id") >= START_TARGET_MONTH_ID) &
        (pl.col("target_month_id") <= END_TARGET_MONTH_ID) &
        (pl.col("y").is_not_null())
    )

    # Fill characteristics by monthly median, then rank-transform to [-1, 1]
    lf = lf.with_columns([
        pl.col(c)
        .fill_null(pl.col(c).median().over("month_id"))
        .fill_null(0.0)
        .cast(pl.Float32)
        .alias(f"{c}_filled")
        for c in CHAR_COLS
    ])

    lf = lf.with_columns(pl.len().over("month_id").alias("_n_month"))

    lf = lf.with_columns([
        (
            2.0
            * (pl.col(f"{c}_filled").rank("average").over("month_id") - 1.0)
            / (pl.col("_n_month") - 1.0)
            - 1.0
        )
        .fill_nan(0.0)
        .cast(pl.Float32)
        .alias(f"{c}_rank")
        for c in CHAR_COLS
    ])

    # Macro columns: fill missing values. Tree models do not require scaling.
    lf = lf.with_columns([
        pl.col(actual_col)
        .fill_null(pl.col(actual_col).median())
        .fill_null(0.0)
        .cast(pl.Float32)
        .alias(f"macro_{macro_name}")
        for macro_name, actual_col in macro_cols.items()
    ])

    # Construct 920 features: char_rank × constant plus char_rank × 8 macro states.
    feature_cols = []
    interaction_exprs = []

    for c in CHAR_COLS:
        base = f"{c}_rank"
        feature_cols.append(base)

        for macro_name in MACRO_NAMES:
            out_name = f"{c}_x_{macro_name}"
            interaction_exprs.append(
                (pl.col(base) * pl.col(f"macro_{macro_name}"))
                .cast(pl.Float32)
                .alias(out_name)
            )
            feature_cols.append(out_name)

    lf = lf.with_columns(interaction_exprs)

    print("Number of features:", len(feature_cols))
    if len(feature_cols) != 920:
        raise ValueError(f"Expected 920 features, got {len(feature_cols)}")

    return lf, feature_cols


def collect_xy(lf, feature_cols, train_max_year=None, test_year=None):
    q = lf

    if train_max_year is not None:
        q = q.filter(pl.col("target_year") <= train_max_year)

    if test_year is not None:
        q = q.filter(pl.col("target_year") == test_year)

    id_cols = [
        "permno", "month", "target_month", "month_id",
        "target_month_id", "target_year", "y", "size_for_ranking"
    ]

    df = q.select(id_cols + feature_cols).collect()

    ids = df.select(id_cols)
    X = df.select(feature_cols).to_numpy().astype(np.float32, copy=False)
    y = df.get_column("y").to_numpy().astype(np.float32, copy=False)

    return ids, X, y


In [ ]:
def run_rf():
    lf, feature_cols = prepare_panel()

    all_preds = []

    for test_year in tqdm(TEST_YEARS, desc=MODEL_NAME):
        year_start = time.perf_counter()

        # GKX-style timing: 12-year validation gap is not used for fitting here.
        # Example: test 1987 trains through 1974; 1975-1986 is left out.
        train_end_year = test_year - 13

        ids_train, X_train, y_train = collect_xy(
            lf,
            feature_cols,
            train_max_year=train_end_year
        )

        ids_test, X_test, y_test = collect_xy(
            lf,
            feature_cols,
            test_year=test_year
        )

        print(
            f"\nYear {test_year}: train_end={train_end_year}, "
            f"X_train={X_train.shape}, X_test={X_test.shape}"
        )

        model = XGBRFRegressor(
            objective="reg:squarederror",
            tree_method="hist",
            n_estimators=200,       # use 300 later if runtime is acceptable
            max_depth=3,            # GKX tune shallow trees; 3 is a safe fast start
            subsample=0.70,
            colsample_bytree=0.70,
            colsample_bynode=0.20,  # random subset of features at each split
            min_child_weight=500,
            reg_lambda=10.0,
            max_bin=32,
            n_jobs=2,               # increase to 4 only if Colab remains stable
            random_state=42,
            verbosity=1,
        )

        model.fit(X_train, y_train)
        pred = model.predict(X_test).astype(np.float32)

        out = ids_test.with_columns([
            pl.Series("prediction", pred),
            pl.lit(MODEL_NAME).alias("model"),
            pl.lit(train_end_year).alias("train_end_year"),
        ])

        year_pred_path = OUTPUT / f"predictions_{MODEL_NAME}_{TARGET_MODE}_{test_year}.parquet"
        out.write_parquet(year_pred_path, compression="zstd")
        print(f"Saved yearly predictions: {year_pred_path}")
        print(f"Year {test_year} R2 pct:", 100 * oos_r2(y_test, pred))
        print(f"Elapsed minutes for year {test_year}:", (time.perf_counter() - year_start) / 60)

        all_preds.append(out)

        del ids_train, X_train, y_train, ids_test, X_test, y_test, pred, out, model
        gc.collect()

    preds = pl.concat(all_preds)

    pred_path = OUTPUT / f"predictions_{MODEL_NAME}_{TARGET_MODE}.parquet"
    preds.write_parquet(pred_path, compression="zstd")

    r2_all = oos_r2(
        preds.get_column("y").to_numpy(),
        preds.get_column("prediction").to_numpy()
    )

    preds_ranked = (
        preds
        .filter(pl.col("size_for_ranking").is_not_null())
        .with_columns([
            pl.col("size_for_ranking")
            .rank("ordinal", descending=True)
            .over("target_month_id")
            .alias("rank_big"),

            pl.col("size_for_ranking")
            .rank("ordinal", descending=False)
            .over("target_month_id")
            .alias("rank_small"),
        ])
    )

    big = preds_ranked.filter(pl.col("rank_big") <= 1000)
    small = preds_ranked.filter(pl.col("rank_small") <= 1000)

    r2_big = oos_r2(
        big.get_column("y").to_numpy(),
        big.get_column("prediction").to_numpy()
    )

    r2_small = oos_r2(
        small.get_column("y").to_numpy(),
        small.get_column("prediction").to_numpy()
    )

    summary = pl.DataFrame({
        "model": [MODEL_NAME],
        "target_mode": [TARGET_MODE],
        "r2_all_pct": [100 * r2_all],
        "r2_top1000_pct": [100 * r2_big],
        "r2_bottom1000_pct": [100 * r2_small],
        "n_obs": [preds.height],
        "n_features": [len(feature_cols)],
        "prediction_file": [str(pred_path)],
    })

    with pl.Config(tbl_cols=-1, tbl_width_chars=180):
        print(summary)

    summary.write_csv(OUTPUT / f"r2_summary_{MODEL_NAME}_{TARGET_MODE}.csv")


In [ ]:
if __name__ == "__main__":
    run_rf()


In [ ]:
MODEL_TO_DIAG = MODEL_NAME

preds = pl.read_parquet(
    OUTPUT / f"predictions_{MODEL_TO_DIAG}_{TARGET_MODE}.parquet"
)

diag = (
    preds
    .with_columns(pl.col("target_month").dt.year().alias("year"))
    .group_by("year")
    .agg([
        pl.len().alias("n"),
        pl.col("y").mean().alias("y_mean"),
        pl.col("prediction").mean().alias("pred_mean"),
        pl.col("y").std().alias("y_std"),
        pl.col("prediction").std().alias("pred_std"),
        pl.corr("y", "prediction").alias("corr_y_pred"),
        ((pl.col("y") - pl.col("prediction")) ** 2).sum().alias("sse"),
        (pl.col("y") ** 2).sum().alias("sst_zero"),
    ])
    .with_columns(
        (100 * (1 - pl.col("sse") / pl.col("sst_zero"))).alias("r2_pct")
    )
    .sort("year")
)

with pl.Config(tbl_cols=-1, tbl_width_chars=180):
    print(
        diag.select([
            "year",
            "n",
            "y_mean",
            "pred_mean",
            "y_std",
            "pred_std",
            "corr_y_pred",
            "r2_pct",
        ])
    )


In [ ]:
print(
    preds.select([
        pl.col("y").quantile(0.01).alias("y_p01"),
        pl.col("y").quantile(0.50).alias("y_p50"),
        pl.col("y").quantile(0.99).alias("y_p99"),
        pl.col("prediction").quantile(0.01).alias("pred_p01"),
        pl.col("prediction").quantile(0.50).alias("pred_p50"),
        pl.col("prediction").quantile(0.99).alias("pred_p99"),
    ])
)
